# [LAB] MCP 서버와 클라이언트 실습

우리가 만든 MCP 서버와 남이 만든 MCP 서버를 한 Agent에 함께 연결합니다.

- **2장 DART 공시 MCP 서버 만들고 띄우기**: 금융감독원 전자공시(DART) API를 MCP 도구 세 개로 감싼 서버 코드를 셀로 나눠 읽고, 터미널에서 띄웁니다. 서버 파일에서 도구 설명 한 줄을 바꿔 봅니다.
- **3장 클라이언트로 연결하기**: 우리 DART 서버와 Tavily가 운영하는 웹 검색 MCP 서버에 함께 연결해 도구를 가져오고, 그 도구로 Agent 그래프를 만듭니다. 보기를 골라 코드를 완성합니다. 마지막에 연결 방식을 SSE에서 stdio로 한 줄 바꿔 비교합니다.

**시작 전에: API 키 두 개 넣기**

- **DART**: 강사가 알려 주는 40자리 인증키를 `.env`의 `DART_API_KEY=` 뒤에 붙여 넣습니다.
- **Tavily** (각자, 무료): `app.tavily.com`에 가입하면 첫 화면에 `tvly-`로 시작하는 키가 있습니다. `.env`의 `TAVILY_API_KEY=` 뒤에 붙여 넣습니다.
- `.env`를 고쳤으면 커널을 다시 시작합니다. 2장의 서버를 이미 띄웠다면 서버도 Ctrl + C로 끄고 다시 띄웁니다.
- 키가 없거나 사내망에서 연결되지 않아도 실습은 끝까지 됩니다. DART 도구는 `[모의 DART]`로 시작하는 연습용 답을 돌려주고, Tavily는 빼고 진행합니다.

셀은 위에서부터 차례로 실행합니다. 3장은 2장의 서버가 터미널에 켜져 있어야 동작합니다.

**목차**

- 1. 실습 방법
- 2. DART 공시 MCP 서버 만들고 띄우기
  - 2.1 서버 만들기
  - 2.2 DART API 호출하기
  - 2.3 Tool 정의하기
  - 2.4 Resource 정의하기
  - 2.5 Prompt 정의하기
  - 2.6 실행 방식 정하기
- 3. 클라이언트로 연결하기
  - 3.1 연결할 서버 정하기
  - 3.2 클라이언트 만들기
  - 3.3 도구 목록 가져오기
  - 3.4 LLM과 도구 노드에 도구 연결하기
  - 3.5 agent 노드
  - 3.6 그래프 연결과 컴파일
  - 3.7 질문하기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["MCP_KEYS"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. DART 공시 MCP 서버 만들고 띄우기

금융감독원 전자공시(DART)의 공개 API를 MCP 도구로 감싼 서버를 만듭니다. DART는 상장회사가 낸 공시와 사업보고서를 누구나 볼 수 있게 내놓는 곳입니다. 서버는 노트북 셀 안에서 돌리지 않고 터미널에서 띄웁니다. 서버는 켜 둔 동안 계속 요청을 기다리기 때문에, 노트북 셀에서 돌리면 다른 셀을 실행할 수 없습니다.

이 폴더에는 완성된 서버 파일 `mcp_server.py`가 미리 들어 있습니다. 아래 셀은 그 파일의 코드를 나눠 보여 주는 곳이라 차례로 읽고 실행해 봅니다. 셀을 실행해도 서버 파일은 바뀌지 않고, DART도 아직 호출하지 않습니다.

서버에는 3대 Primitive가 모두 들어 있습니다.

- **Tool 3개:** 회사 기본 정보, 최근 공시 목록, 한 해 실적(매출액·영업이익·당기순이익)
- **Resource 1개:** 찾을 수 있는 회사 목록(`dart://companies`)
- **Prompt 1개:** 회사 공시 브리핑 요청 템플릿

`DART_API_KEY`가 없거나 DART에 연결되지 않으면 도구는 `[모의 DART]`로 시작하는 연습용 답을 돌려줍니다. 실제 공시가 아니라는 표시이니 그 답의 숫자는 믿지 않습니다.

셀을 다 본 뒤, 아래 안내대로 터미널에서 서버를 띄웁니다.

### 2.1 서버 만들기

In [ ]:
"""
DART 공시 MCP 서버 만들기 (FastMCP)

금융감독원 전자공시(OpenDART) API를 MCP 도구로 감싼다. 터미널에서 실행한다.
기본은 HTTP/SSE 모드(포트 8000), --stdio를 붙이면 stdio 모드로 실행한다.
.env에 DART_API_KEY가 없거나 DART에 연결되지 않으면 [모의 DART]로 시작하는 연습용 답을 돌려준다.
"""

import os
import sys
from datetime import date, timedelta

import requests
import truststore
from dotenv import load_dotenv
from fastmcp import FastMCP

truststore.inject_into_ssl()  # 사내망 HTTPS는 회사 인증서를 거친다. PC에 등록된 인증서를 믿게 한다
load_dotenv()

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP("DART 공시 서버")

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(서버 객체만 만들어 출력은 없다)

### 2.2 DART API 호출하기

In [ ]:
# ── DART API 호출하기 ──────────────────────────────────────────────────────────
DART_URL = "https://opendart.fss.or.kr/api"
# DART는 회사를 8자리 고유번호(corp_code)로 찾는다. 실습에서 쓸 회사만 사전으로 둔다
COMPANIES = {
    "삼성전자": "00126380",
    "SK하이닉스": "00164779",
    "LG전자": "00401731",
    "현대자동차": "00164742",
    "NAVER": "00266961",
    "카카오": "00258801",
}
MOCK = "[모의 DART] DART_API_KEY가 없거나 DART에 연결하지 못해 연습용 예시를 보여 줍니다. 실제 공시가 아닙니다."


def call_dart(endpoint: str, **params) -> dict | None:
    """DART API를 호출한다. 키가 없거나 DART에 연결하지 못하면 None을 돌려준다(그때 도구는 [모의 DART] 답을 낸다)."""
    key = os.getenv("DART_API_KEY", "").strip()
    if not key:
        return None
    try:
        return requests.get(
            f"{DART_URL}/{endpoint}", params={"crtfc_key": key, **params}, timeout=10
        ).json()
    except (requests.RequestException, ValueError):
        return None


def find_code(company: str) -> str | None:
    return COMPANIES.get(company.replace(" ", "").replace("(주)", ""))


def unknown(company: str) -> str:
    return f"'{company}'은(는) 이 서버의 회사 사전에 없습니다. 찾을 수 있는 회사: {', '.join(COMPANIES)}"

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(함수만 만들어 출력은 없다)

### 2.3 Tool 정의하기

**✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다**

먼저 그대로 두고 3장의 도구 목록까지 실행해 봅니다. 서버는 이 셀이 아니라 폴더의 `mcp_server.py` 파일로 돌기 때문에, 바꿀 때는 그 파일을 고칩니다. VS Code 왼쪽 탐색기에서 노트북과 같은 폴더의 `mcp_server.py`를 열고, `get_recent_disclosures` 함수 아래 따옴표 안 설명을 바꿔(예: "회사의 최근 공시 제목을 날짜순으로 알려 준다.") 저장합니다(`Ctrl + S`). 터미널에서 **Ctrl + C**로 서버를 끄고 같은 명령으로 다시 띄운 뒤, 3장의 3.1 셀부터 도구 목록 셀까지 다시 실행합니다. 서버의 도구 설명은 클라이언트가 가져가 LLM이 도구를 고르는 근거로 씁니다.

In [ ]:
# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_company_info(company: str) -> str:
    """회사의 기본 정보(정식 이름, 대표이사, 종목코드, 설립일, 홈페이지)를 DART에서 찾는다."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    data = call_dart("company.json", corp_code=code)
    if data is None:
        return f"{MOCK}\n회사명: {company}(예시) / 대표이사: 홍길동 / 종목코드: 000000 / 설립일: 19690101"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')}"
    return (
        f"회사명: {data['corp_name']} / 대표이사: {data['ceo_nm']} / 종목코드: {data['stock_code']} / "
        f"설립일: {data['est_dt']} / 홈페이지: {data['hm_url']}"
    )


@mcp.tool()
def get_recent_disclosures(company: str, days: int = 30) -> str:
    # ✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다
    """회사가 최근 며칠 동안 DART에 낸 공시 목록(날짜, 공시 제목, 제출인)을 찾는다. 기본은 30일."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    end = date.today()
    begin = end - timedelta(days=days)
    data = call_dart(
        "list.json",
        corp_code=code,
        bgn_de=begin.strftime("%Y%m%d"),
        end_de=end.strftime("%Y%m%d"),
        page_count=10,
    )
    if data is None:
        return f"{MOCK}\n{end} 주요사항보고서(예시) / {begin} 분기보고서(예시)"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')}"
    day = lambda d: f"{d[:4]}-{d[4:6]}-{d[6:]}"  # 20261019 → 2026-10-19
    rows = [
        f"{day(x['rcept_dt'])} {x['report_nm'].strip()} (제출: {x['flr_nm']})"
        for x in data["list"]
    ]
    return (
        f"{data['list'][0]['corp_name']} 최근 {days}일 공시 {data['total_count']}건 중 최근 {len(rows)}건:\n"
        + "\n".join(rows)
    )


@mcp.tool()
def get_financial_summary(company: str, year: int) -> str:
    """회사의 한 해 실적(연결 기준 매출액, 영업이익, 당기순이익)을 그해 사업보고서에서 찾는다."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    data = call_dart(
        "fnlttSinglAcnt.json", corp_code=code, bsns_year=str(year), reprt_code="11011"
    )
    if data is None:
        return f"{MOCK}\n{year}년 매출액 100.0조 원 / 영업이익 10.0조 원 / 당기순이익 8.0조 원(예시)"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')} ({year}년 사업보고서)"
    lines = []
    for x in data["list"]:
        if x["fs_div"] == "CFS" and x["account_nm"] in (
            "매출액",
            "영업이익",
            "당기순이익(손실)",
            "당기순이익",
        ):
            now, before = (
                int(v.replace(",", "")) / 1e12
                for v in (x["thstrm_amount"], x["frmtrm_amount"])
            )
            lines.append(f"{x['account_nm']}: {now:.1f}조 원 (전년 {before:.1f}조 원)")
    return f"{company} {year}년 연결 실적\n" + "\n".join(
        dict.fromkeys(lines)
    )  # 같은 줄이 두 번 오는 회사가 있다

**성공 기준**

- 🟢 셀이 오류 없이 실행된다
- 🟢 (파일의 설명을 바꾼 뒤) 서버를 다시 띄우고 3장 도구 목록을 다시 출력하면 `get_recent_disclosures` 옆 설명이 바꾼 글로 보인다
- ⛔ 설명이 그대로다 → `mcp_server.py` 파일을 저장했는지, 서버를 Ctrl + C로 끄고 다시 띄웠는지, 3장 도구 목록 셀을 다시 실행했는지 봅니다

### 2.4 Resource 정의하기

In [ ]:
# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource("dart://companies")
def list_companies() -> str:
    """이 서버가 찾을 수 있는 회사와 DART 고유번호 목록."""
    return "\n".join(f"{name}: {code}" for name, code in COMPANIES.items())

### 2.5 Prompt 정의하기

In [ ]:
# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def company_briefing(company: str) -> str:
    """회사 한 곳의 공시 브리핑을 요청하는 프롬프트를 만든다."""
    return f"{company}의 기본 정보, 최근 30일 공시, 지난해 실적을 DART 도구로 확인하고 꼭 알아야 할 점을 세 줄로 정리해줘."

### 2.6 실행 방식 정하기

아래는 서버를 터미널에서 띄울 때만 실행되는 부분입니다. 노트북에서는 실행하지 않고 읽어만 봅니다.

```python
if __name__ == "__main__":
    if "--stdio" in sys.argv:
        # stdio 모드: 클라이언트가 이 파일을 직접 실행해 표준 입출력으로 주고받는다(print를 쓰면 통신이 깨진다)
        mcp.run(transport="stdio")
    else:
        print("MCP 서버 시작: http://localhost:8000/sse")
        print(
            "DART_API_KEY:",
            "있음 — 실제 DART를 호출합니다"
            if os.getenv("DART_API_KEY", "").strip()
            else "없음 — [모의 DART] 연습용 답을 돌려줍니다",
        )
        print("종료하려면 Ctrl+C")
        # HTTP/SSE 모드: 클라이언트가 localhost:8000으로 연결
        # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
        mcp.run(transport="sse", host="127.0.0.1", port=8000)
```

이 폴더에는 정답이 모두 들어간 서버 파일(`mcp_server.py`)이 미리 들어 있습니다. 2.1부터 위 셀들은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀을 실행해 오류가 없는지 확인합니다(보기가 있는 셀은 먼저 고릅니다). 셀이 파일을 고치지는 않습니다.

서버는 터미널에서 이 완성 파일로 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/mcp_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다. 서버를 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

**성공 기준** (터미널 화면)

- 🟢 `MCP 서버 시작: http://localhost:8000/sse`, `DART_API_KEY: 있음 — 실제 DART를 호출합니다`, `종료하려면 Ctrl+C` 세 줄이 보인다(키가 없으면 둘째 줄이 `DART_API_KEY: 없음 — [모의 DART] 연습용 답을 돌려줍니다`)
- 🟢 그 아래 FastMCP 안내 상자(서버 이름 `DART 공시 서버`가 적힌 상자)가 나오고, 맨 끝 줄에 `Uvicorn running on`과 `(Press CTRL+C to quit)`가 보인다
- 🟢 명령이 끝나지 않고 기다린다(새 명령을 입력하는 줄이 돌아오지 않는다)
- ⛔ 명령이 곧바로 끝나고 오류가 나온다 → 오류 메시지 전체를 AI에게 보여 줍니다
- ⛔ 오류에 `address already in use` 또는 `10048`이 보인다 → 이미 켜 둔 서버가 있습니다. 앞서 서버를 띄운 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

**볼 지점**
- 서버를 띄운 터미널은 3장이 끝날 때까지 켜 둡니다. 3장에서 DART 도구를 호출할 때마다 이 터미널에 요청 기록(`POST /messages/...` 같은 줄)이 늘어납니다.
- 서버는 시작할 때 `.env`를 읽습니다. DART 키를 나중에 넣었다면 이 서버를 Ctrl+C로 끄고 다시 띄웁니다.
- 안내 상자에 새 버전 안내(`Update available`)가 함께 나올 수 있습니다. 실습과 관계없으니 그대로 둡니다.

## 3. 클라이언트로 연결하기

노트북에서 Agent(Host)를 만들고, 그 안의 Client로 MCP 서버 두 곳에 연결합니다. 하나는 2장에서 띄운 우리 DART 서버이고, 다른 하나는 Tavily가 인터넷에 운영하는 웹 검색 MCP 서버입니다. 두 서버의 도구를 한 목록으로 가져와 LLM에 넘기고, 과정 1에서 만든 것과 같은 모양의 LangGraph 그래프(agent ⇄ tools)로 질문에 답하게 합니다.

- **연결:** 클라이언트 만들기, 도구 목록 가져오기
- **Agent 그래프:** LLM과 도구 노드에 도구 연결하기, agent 노드, 조건부 엣지, 컴파일

서버와 주고받는 코드 앞에는 `await`가 붙습니다. 응답이 올 때까지 기다린다는 표시입니다. 노트북 셀에서는 `await`를 바로 쓸 수 있습니다.

마지막 질문 셀은 앞의 셀에서 **모두 보기를 골라 실행한 뒤에** 동작합니다.

**왜 배우나:** 남이 만든 서버든 내가 만든 서버든 연결 설정만 다르고, 받아 온 도구는 똑같이 과정 1의 `bind_tools`·ToolNode·조건부 엣지로 씁니다. MCP가 Agent에 도구를 꽂아 쓰는 표준 단자라는 것을 직접 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고, 설정 파일을 읽고, MCP 서버에 닿는지 확인하는 함수를 만듭니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
MCP 클라이언트 실습: 터미널에 켜 둔 DART MCP 서버(우리가 만든 서버)와 Tavily 공식 MCP 서버(남이 만든 서버)에
연결해 LangGraph Agent가 두 서버의 도구를 함께 쓴다.
"""

import asyncio
import os
import sys
from typing import Annotated

import truststore
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm
from config.mcp import stdio_errors_to_file

truststore.inject_into_ssl()  # 사내망에서 Tavily(인터넷 HTTPS)에 연결할 때 회사 인증서를 믿게 한다
load_dotenv()


async def reachable(connection: dict) -> bool:
    """이 연결 설정으로 MCP 서버에 10초 안에 닿으면 True. 사내망에서 막힌 서버는 빼고 진행하려고 쓴다."""
    try:
        await asyncio.wait_for(
            MultiServerMCPClient({"check": connection}).get_tools(), timeout=10
        )
        return True
    except Exception:  # 연결 실패는 여러 오류가 한데 묶여(ExceptionGroup) 올라온다
        return False

### 3.1 연결할 서버 정하기

**✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다**

처음에는 그대로(`"sse"`) 두고 3장 끝까지 실행합니다. 모두 동작하면 이 줄을 `"stdio"`로 바꾸고 이 셀부터 질문하기 셀까지 차례로 다시 실행합니다. SSE는 터미널에 켜 둔 서버에 주소로 접속하고, stdio는 노트북이 서버 파일을 직접 실행해 표준 입출력으로 주고받습니다. 그래서 stdio에서는 2장의 서버 터미널을 Ctrl + C로 꺼 두어도 도구를 그대로 씁니다. Tavily 서버는 바꾸지 않고 그대로 인터넷 주소로 접속합니다. stdio로 띄운 서버의 기록은 노트북 폴더의 `mcp_stdio_server.log`에 쌓입니다. 비교가 끝나면 `"sse"`로 되돌립니다.

In [ ]:
# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# dart: 2장에서 터미널에 켜 둔 우리 서버. SSE는 켜 둔 서버에 주소로 접속하고,
#       stdio는 노트북이 서버 파일을 직접 실행해 표준 입출력으로 주고받는다.
#       stdio로 띄운 서버의 오류 출력은 화면 대신 mcp_stdio_server.log 파일로 받는다
#       (Windows 노트북의 화면 출력은 진짜 파일이 아니라, 그대로 두면 서버를 띄우지 못한다)
stdio_errors_to_file("mcp_stdio_server.log")
CONNECTIONS = {
    "sse": {"transport": "sse", "url": "http://localhost:8000/sse"},
    "stdio": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["mcp_server.py", "--stdio"],
    },
}
# ✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다
TRANSPORT = "sse"
MCP_CONFIG = {"dart": CONNECTIONS[TRANSPORT]}

# tavily: Tavily가 인터넷에 운영하는 공식 MCP 서버(남이 만든 서버). HTTP로 접속하고, 키는 주소가 아니라
#         머리말(headers)에 넣는다(오류 메시지에 키가 찍히지 않게). 키가 있고 연결될 때만 더한다
TAVILY = {
    "transport": "streamable_http",
    "url": "https://mcp.tavily.com/mcp/",
    "headers": {"Authorization": f"Bearer {os.getenv('TAVILY_API_KEY', '').strip()}"},
}
if os.getenv("TAVILY_API_KEY", "").strip() and await reachable(TAVILY):
    MCP_CONFIG["tavily"] = TAVILY
print("사용할 MCP 서버:", ", ".join(MCP_CONFIG))
if "tavily" not in MCP_CONFIG:
    print(
        "Tavily는 뺐습니다(TAVILY_API_KEY가 없거나 Tavily 서버에 연결되지 않음). DART 도구만으로 진행합니다."
    )

**성공 기준**

- 🟢 `사용할 MCP 서버: dart, tavily`가 나온다(Tavily 키가 없거나 연결되지 않으면 `사용할 MCP 서버: dart`와 `Tavily는 뺐습니다…` 두 줄이 나오고, 그대로 진행한다)
- 🟢 ("stdio"로 바꾸고 다시 실행한 뒤) 서버 터미널을 Ctrl + C로 꺼 두어도 도구 목록에 같은 DART 도구 3개가 나오고 질문에 답한다(Windows도 같다)
- ⛔ ("stdio"로 바꾼 뒤) 도구 목록 가져오기 셀에서 오류가 난다 → 노트북 폴더의 `mcp_stdio_server.log` 마지막 줄과 오류 메시지 전체를 AI에게 보여 원인을 묻습니다. 풀리지 않으면 `"sse"`로 되돌려 넘어갑니다

### 3.2 클라이언트 만들기

**✅ MCP 설정으로 MultiServerMCPClient를 생성해 주세요.**

- 🟢 3.3 도구 목록 가져오기 셀까지 오류 없이 DART 도구 3줄(Tavily가 연결되면 `tavily_search`까지 4줄)이 나온다
- ⛔ 3.2에서 `TypeError`가 나거나 3.3에서 `AttributeError`·`ValueError: Configuration error`가 보인다 → 다른 보기를 고릅니다

In [ ]:
# ✅ CHOOSE_HERE: MCP 설정으로 MultiServerMCPClient를 생성해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# client = MultiServerMCPClient(TRANSPORT)           # 1️⃣ 연결 방식 글자를 넘김
# client = await MultiServerMCPClient(MCP_CONFIG)    # 2️⃣ 클라이언트 만들기를 기다림
# client = MultiServerMCPClient(MCP_CONFIG)          # 3️⃣ 서버 이름이 붙은 설정을 넘김
# client = MultiServerMCPClient(MCP_CONFIG["dart"])  # 4️⃣ dart 서버 하나의 연결 설정만 넘김

<details><summary>정답 보기</summary>

위 셀에서 만든 연결 설정 `MCP_CONFIG`로 클라이언트를 만들어 변수 `client`에 담는 한 줄입니다. 이 줄은 서버에 바로 접속하지 않습니다. 설정만 들고 있다가 다음 셀에서 도구를 가져올 때 접속합니다.

**정답: 3️⃣**

- 1️⃣ `TRANSPORT`는 `"sse"`라는 글자일 뿐입니다. 3.2는 통과하지만 3.3에서 도구를 가져올 때 `AttributeError: 'str' object has no attribute 'items'`가 납니다.
- 2️⃣ 클라이언트를 만드는 일은 서버와 주고받지 않아 기다릴 것이 없습니다. 3.2에서 `TypeError: object MultiServerMCPClient can't be used in 'await' expression`이 납니다.
- 3️⃣ `MCP_CONFIG`는 서버 이름(`dart`, 연결되면 `tavily`)과 각각의 연결 설정을 묶은 것입니다. 클라이언트는 이 설정을 들고 있다가 3.3에서 도구를 가져올 때 두 서버에 접속합니다.
- 4️⃣ 서버 이름 없이 dart 서버의 연결 설정만 넘겨 `transport`·`url`을 서버 이름으로 읽습니다. 3.3에서 `ValueError: Configuration error: Missing 'transport' key in server configuration`이 납니다.

</details>

### 3.3 도구 목록 가져오기

**✅ MCP 서버에서 도구 목록을 가져와 주세요.**

- 🟢 `=== 사용 가능한 MCP 도구 ===` 아래에 `- get_company_info: …`로 시작하는 DART 도구 3줄이 나온다(Tavily가 연결되면 `- tavily_search: …`까지 4줄)
- ⛔ `TypeError`나 `AttributeError`가 보인다 → 다른 보기를 고릅니다(`ConnectError`면 2장의 서버 터미널을 켭니다)

In [ ]:
# get_tools()는 서버에 연결해 도구 목록을 가져온다.
# ✅ CHOOSE_HERE: MCP 서버에서 도구 목록을 가져와 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# tools = await client.get_tools()            # 1️⃣ 도구 목록을 받아 올 때까지 기다림
# tools = client.get_tools()                  # 2️⃣ await 없이 부름
# tools = await client.get_tools              # 3️⃣ 괄호 없이 기다림
# tools = await client.get_resources("dart")  # 4️⃣ 서버의 Resource를 가져옴
# Tavily 도구는 검색(tavily_search)만 쓴다. 나머지 넷(extract·crawl·map·research)은 크레딧을 많이 쓴다
tools = [
    t for t in tools if not t.name.startswith("tavily_") or t.name == "tavily_search"
]
print("=== 사용 가능한 MCP 도구 ===")
for t in tools:
    print(f"  - {t.name}: {t.description.strip().splitlines()[0][:70]}")
print()

<details><summary>정답 보기</summary>

연결 설정에 든 서버(dart, 연결되면 tavily)에 접속해 도구 이름과 설명을 읽어 LangChain Tool 목록으로 받아 `tools`에 담는 한 줄입니다. 서버와 주고받는 동안 기다려야 하므로 `await`가 필요합니다. 바로 아래 줄은 Tavily 도구 가운데 검색(`tavily_search`)만 남기고, 그 아래 줄들이 받은 도구를 하나씩 출력합니다.

**정답: 1️⃣**

- 1️⃣ 연결 설정에 든 서버에 접속해 도구를 받아 옵니다(DART 3개, Tavily가 연결되면 5개 더). 서버와 주고받는 일이라 `await`로 응답을 기다립니다. 바로 아래 줄이 Tavily 도구 가운데 `tavily_search`만 남깁니다.
- 2️⃣ 응답을 기다리지 않아 도구 목록 대신 아직 실행되지 않은 작업(coroutine)이 담깁니다. 바로 아래 줄에서 도구 목록이 나오기 전에 `TypeError: 'coroutine' object is not iterable`이 납니다.
- 3️⃣ 괄호가 없어 함수를 부르지 않고 함수 자체를 기다립니다. `TypeError: object method can't be used in 'await' expression`이 납니다.
- 4️⃣ 도구가 아니라 dart 서버의 Resource(`dart://companies`, 회사 목록)를 가져옵니다. Resource는 데이터라서 도구 이름이 없습니다. 바로 아래 줄에서 도구 목록이 나오기 전에 `AttributeError: 'Blob' object has no attribute 'name'`이 납니다.

</details>

### 3.4 LLM과 도구 노드에 도구 연결하기

**✅ 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.**

- 🟢 3.7 실적 질문의 답에 SK하이닉스 2025년 매출액 97.1조 원과 영업이익 47.2조 원이 들어 있다(키가 없으면 `[모의 DART]` 예시 숫자)
- ⛔ 3.7에서 DART를 찾지 않고 "확정되지 않았다"·"확인할 수 없다"고 답하거나 `AttributeError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
# ✅ CHOOSE_HERE: 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# llm = get_llm()                    # 1️⃣ 도구 노드만 만듦
# tool_node = ToolNode(tools)        # 1️⃣
#
# llm = get_llm().bind_tools(tools)  # 2️⃣ ToolNode를 도구 없이 그대로 씀
# tool_node = ToolNode               # 2️⃣
#
# llm = get_llm().bind_tools(tools)  # 3️⃣ 빈 목록으로 도구 노드를 만듦
# tool_node = ToolNode([])           # 3️⃣
#
# llm = get_llm().bind_tools(tools)  # 4️⃣ 모델과 도구 노드에 같은 도구를 줌
# tool_node = ToolNode(tools)        # 4️⃣

<details><summary>정답 보기</summary>

두 줄입니다. 하나는 `get_llm()`으로 만든 모델에 서버의 도구 목록을 알려 주어 `llm`에 담고, 다른 하나는 같은 도구 목록으로 도구를 실제로 실행하는 노드 `tool_node`를 만듭니다. 과정 1의 Tool Calling과 LangGraph에서 쓴 방법과 같습니다.

**정답: 4️⃣**

- 1️⃣ 모델이 도구가 있는지 모릅니다. 오류 없이 3.7에서 실적 질문에 DART를 찾지 않고 "2025년 실적은 아직 확정되지 않았다"처럼 모델이 알고 있던 지식만으로 답합니다.
- 2️⃣ `ToolNode`에 도구 목록을 넘기지 않아 노드가 만들어지지 않습니다. 3.7 첫 질문에서 `AttributeError: 'function' object has no attribute 'name'`이 납니다.
- 3️⃣ 모델은 도구를 부르지만 도구 노드에 실행할 도구가 없습니다. 오류 없이 3.7에서 도구 호출이 여러 번 되풀이된 뒤 "재무 조회 도구에 접근할 수 없어 확인할 수 없다"는 식으로 답합니다.
- 4️⃣ 모델은 어떤 도구를 부를 수 있는지 알고, 도구 노드는 그 도구를 실제로 실행합니다. 3.7에서 공시 목록, SK하이닉스 2025년 매출액 97.1조 원·영업이익 47.2조 원, 뉴스와 공시를 묶은 세 줄 요약이 나옵니다.

</details>

### 3.5 agent 노드

**✅ State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

- 🟢 3.7에서 질문마다 마지막에 답이 든 `Ai Message` 칸이 나온다
- ⛔ 3.7에서 `Ai Message` 칸 없이 `Human Message`만 나오거나 `ValueError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    # ✅ CHOOSE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # response = llm.invoke(state)              # 1️⃣ State 전체를 LLM에 넘김
    # return {"messages": [response]}           # 1️⃣
    #
    # response = llm.invoke(state["messages"])  # 2️⃣ 메시지 목록을 넘기고 응답을 담음
    # return {"messages": [response]}           # 2️⃣
    #
    # response = llm.invoke(state["messages"])  # 3️⃣ 받은 State를 그대로 돌려줌
    # return state                              # 3️⃣
    #
    # response = llm.invoke(state["messages"])  # 4️⃣ 응답의 글자만 담음
    # return {"messages": [response.content]}   # 4️⃣
    pass  # 그대로 둡니다

<details><summary>정답 보기</summary>

`agent_node`는 Agent가 한 번 생각하는 단계입니다. 두 줄입니다. 상태에 쌓인 메시지를 도구가 연결된 `llm`에 보내 응답을 받고, 그 응답을 `messages`에 더하도록 딕셔너리로 돌려줍니다.

**정답: 2️⃣**

- 1️⃣ State 딕셔너리를 통째로 넘깁니다. LLM은 메시지 목록을 받아야 해서 3.7에서 `ValueError: Invalid input type <class 'dict'>`가 납니다.
- 2️⃣ 메시지 목록을 넘겨 응답을 받고, 대괄호로 감싸 `messages`에 담습니다. 응답에 도구 호출 요청이 들어 있으면 조건부 엣지가 tools로 보냅니다.
- 3️⃣ 응답을 버리고 받은 State를 그대로 돌려줍니다. 오류 없이 3.7에 `Ai Message` 칸 없이 질문(`Human Message`)만 나옵니다.
- 4️⃣ 응답의 글자만 담아 도구 호출 요청이 사라집니다. 글자만 담으면 `Human Message`로 쌓이고, 도구를 부를 때는 그 글자가 비어 있어서 오류 없이 3.7에 빈 `Human Message` 칸이 나오고 끝납니다.

</details>

### 3.6 그래프 연결과 컴파일

**✅ tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

- 🟢 3.7에서 공시 질문의 마지막 `Ai Message`에 최근 공시(날짜·제목)를 정리한 답이 나온다
- ⛔ 3.7이 `Tool Message` 칸이나 `Tool Calls:`가 든 `Ai Message` 칸에서 끝나거나 3.6에서 `TypeError`가 보인다 → 다른 보기를 고릅니다

**✅ 그래프를 컴파일해 주세요.**

- 🟢 3.7이 오류 없이 세 질문에 모두 답한다
- ⛔ 3.6이나 3.7에서 `ValueError`·`TypeError`·`AttributeError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent");
# ✅ CHOOSE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# builder.add_conditional_edges("agent", tools_condition)    # 1️⃣ agent 뒤 분기 + tools→agent
# builder.add_edge("tools", "agent")                         # 1️⃣
#
# builder.add_conditional_edges("agent", tools_condition)    # 2️⃣ agent 뒤 분기만 둠
#
# builder.add_conditional_edges("tools", tools_condition)    # 3️⃣ tools 뒤 분기 + tools→agent
# builder.add_edge("tools", "agent")                         # 3️⃣
#
# builder.add_conditional_edges("agent", tools_condition())  # 4️⃣ tools_condition을 불러 그 결과로 분기
# builder.add_edge("tools", "agent")                         # 4️⃣

# ✅ CHOOSE_HERE: 그래프를 컴파일해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# app = builder.compile              # 1️⃣ 괄호 없이 compile을 담음
# app = await builder.compile()      # 2️⃣ 컴파일을 기다림
# app = builder.compile()            # 3️⃣ 조립한 builder를 컴파일
# app = StateGraph(State).compile()  # 4️⃣ 새 그래프를 컴파일

<details><summary>정답 보기</summary>

**tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

두 줄입니다. agent 노드 다음에 LLM이 도구 호출을 요청했으면 tools로, 아니면 끝(END)으로 가게 조건부 엣지를 답니다. 그리고 tools 노드가 도구를 실행한 뒤에는 다시 agent로 돌아와 결과를 보고 답하게 합니다.

**정답: 1️⃣**

- 1️⃣ agent 뒤에서 도구 호출이 있으면 tools로, 없으면 끝으로 갑니다. 도구를 실행한 뒤 agent로 돌아와 결과를 보고 답합니다.
- 2️⃣ 도구를 실행한 뒤 돌아올 길이 없어 그래프가 끝납니다. 오류 없이 3.7이 `[{'type': 'text', 'text': '삼성전자 최근 30일 공시 …'}]` 같은 도구 결과가 든 `Tool Message` 칸에서 끝납니다.
- 3️⃣ 분기를 agent가 아니라 tools 뒤에 달아 agent 다음에 갈 곳이 없습니다. 도구를 부르기 전에 끝나서, 오류 없이 3.7이 도구 호출(`Tool Calls:`)이 든 `Ai Message` 칸에서 끝납니다.
- 4️⃣ `tools_condition`에는 함수 자체를 넘깁니다. 괄호를 붙이면 그 자리에서 불러 3.6에서 `TypeError: tools_condition() missing 1 required positional argument: 'state'`가 납니다.

**그래프를 컴파일해 주세요.**

노드와 엣지를 다 붙인 `builder`를 실행할 수 있는 그래프로 만들어 `app`에 담는 한 줄입니다. 다음 셀이 `app.ainvoke(...)`로 이 그래프를 실행합니다.

**정답: 3️⃣**

- 1️⃣ 괄호가 없어 컴파일하지 않고 `compile` 함수 자체를 담습니다. 3.7에서 `AttributeError: 'function' object has no attribute 'ainvoke'`가 납니다.
- 2️⃣ 컴파일은 서버와 주고받지 않아 기다릴 것이 없습니다. 3.6에서 `TypeError: object CompiledStateGraph can't be used in 'await' expression`이 납니다.
- 3️⃣ 노드와 엣지를 붙인 설계도(`builder`)를 실행할 수 있는 `app`으로 만듭니다. 3.7에서 `app.ainvoke(...)`로 실행합니다.
- 4️⃣ 노드가 하나도 없는 새 그래프를 컴파일합니다. 3.6에서 `ValueError: Graph must have an entrypoint`가 납니다.

</details>

### 3.7 질문하기

**✍️ CHANGE_HERE: 첫 질문의 회사를 바꿔 봅니다**

보기를 모두 골라 세 질문의 답을 본 뒤에 합니다. `# ✍️ CHANGE_HERE:` 아래 따옴표 안의 `삼성전자`를 서버 사전에 있는 다른 회사(예: `카카오`)로 바꿔 이 셀을 다시 실행하고, 그다음 사전에 없는 회사(예: `쿠팡`)로도 바꿔 봅니다. 사전에 없는 회사는 DART 도구가 "회사 사전에 없습니다"라고 돌려주고, LLM은 그 도구 결과를 보고 답합니다. 답은 도구가 돌려준 내용에 따라 정해진다는 것을 봅니다. 확인했으면 `삼성전자`로 되돌립니다.

In [ ]:
# 도구를 사용해 요청 처리
questions = [
    # ✍️ CHANGE_HERE: 첫 질문의 회사를 바꿔 봅니다
    "삼성전자 최근 30일 공시를 알려 줘.",
    "SK하이닉스의 2025년 매출액과 영업이익은 얼마야?",
    "삼성전자 관련 최신 뉴스를 검색하고, DART 최근 공시와 함께 세 줄로 정리해 줘.",
]

for question in questions:
    mcp_graph_result = await app.ainvoke({"messages": [HumanMessage(content=question)]})
    results = mcp_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

**성공 기준**

- 🟢 (사전에 없는 회사로 바꾼 뒤) `Tool Message` 칸에 `회사 사전에 없습니다`와 찾을 수 있는 회사 목록이 보이고, 마지막 답이 그 내용을 전한다
- ⛔ (바꾼 뒤) 셀이 끝나지 않고 계속 돈다 → 셀 실행을 중지(■)하고 3.6에서 고른 조건부 엣지 보기를 다시 봅니다

**볼 지점** (질문하기 셀)
- 질문마다 `Human Message` → `Ai Message`(도구 호출) → `Tool Message`(서버 도구의 결과) → `Ai Message`(답) 칸이 나옵니다.
  - 공시 질문: `get_recent_disclosures`가 가져온 삼성전자 최근 30일 공시(날짜·제목·제출인)
  - 실적 질문: `get_financial_summary`가 가져온 SK하이닉스 2025년 매출액 97.1조 원, 영업이익 47.2조 원
  - 뉴스 질문: `tavily_search`(뉴스)와 `get_recent_disclosures`(공시)를 함께 호출한 뒤 세 줄로 정리
- `[모의 DART]`로 시작하는 도구 결과가 보이면 실제 공시가 아닙니다. `.env`의 `DART_API_KEY`를 확인하고, 고쳤으면 2장의 서버를 다시 띄웁니다.
- Tavily를 뺀 상태라면 뉴스 질문에 검색 없이 공시만으로 정리하거나 "뉴스를 검색할 도구가 없다"고 답합니다. 정상입니다.
- 서버 터미널에는 DART 도구를 호출할 때마다 요청 기록이 늘어납니다. `tavily_search`는 인터넷의 Tavily 서버에서 실행되어 이 터미널에 기록이 남지 않습니다. 한 Agent가 두 곳의 서버 도구를 함께 쓴다는 뜻입니다.
- 답의 말투와 모양은 실행할 때마다 조금씩 다릅니다.

**stdio로 바꿔 보기**

3.1 셀의 `TRANSPORT`를 `"stdio"`로 바꾸고 질문하기 셀까지 다시 실행해 결과를 SSE 때와 비교합니다(성공 기준은 3.1 셀 아래에 있습니다). Windows에서는 도구 목록 가져오기 셀에서 오류가 나며 멈추는 것이 정상입니다. 그때는 `"sse"`로 되돌리고 아래 설명으로 차이를 봅니다. stdio에서는 노트북이 도구를 호출할 때마다 서버 파일을 새로 실행하고 끝냅니다. 그래서 터미널의 서버를 꺼 두어도 답이 나옵니다. DART 도구는 호출할 때마다 DART에서 새로 읽어 오므로 답의 내용은 SSE 때와 같습니다.

## 4. 정리

**이 노트북에서 한 것**
- DART 공시 API를 Tool 3개·Resource 1개·Prompt 1개로 감싼 MCP 서버 코드를 셀로 나눠 읽고, 터미널에서 HTTP/SSE 모드로 띄웠습니다.
- `MultiServerMCPClient` 하나로 우리 서버(`dart`)와 Tavily가 운영하는 서버(`tavily`)에 함께 연결하고, `get_tools`로 두 서버의 도구를 한 목록으로 가져왔습니다. Resource와 Prompt는 도구 목록에 들어오지 않습니다.
- 가져온 도구를 `bind_tools`와 ToolNode에 넘겨, 과정 1과 같은 agent ⇄ tools 그래프로 질문에 답하게 했습니다. 질문에 따라 LLM이 DART 도구와 Tavily 검색을 골라 호출합니다.
- 연결 방식을 stdio로 바꾸면 노트북이 서버 파일을 직접 실행합니다. stdio로 띄운 서버의 기록은 `mcp_stdio_server.log`에 남습니다.

실습이 끝나면 서버를 띄운 터미널에서 **Ctrl+C**로 서버를 끕니다.